In [1]:
!pip install datasets tqdm pandas
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Kaggle doesn't allow & so use subprocess instead
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull aya:8b




The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (565 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#####

In [2]:
from tqdm.notebook import tqdm
from pathlib import Path
import pandas as pd
import requests
import os

In [3]:
base_path = Path("/kaggle/input/datasets/naghamalshbrawi/samer-simplification/samer-simplification-corpus-v1/data")
OLLAMA_URL = "http://localhost:11434/api/generate"

In [4]:
train_df = pd.read_csv(base_path / "train.tsv", sep="\t", dtype=str)
dev_df   = pd.read_csv(base_path / "dev.tsv",   sep="\t", dtype=str)
test_df  = pd.read_csv(base_path / "test.tsv",  sep="\t", dtype=str)

df_raw = pd.concat([train_df, dev_df, test_df], ignore_index=True)
print(f"Loaded {len(df_raw)} rows")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(3)

Loaded 20603 rows
Columns: ['Novel', 'Chapter', 'L5', 'L4', 'L3', 'Line']


,Novel,Chapter,L5,L4,L3,Line
0,0,2,الفصل الثاني,الفصل الثاني,الفصل الثاني,0
1,0,2,«وكان صباح..,«وكان صباح..,«وكان صباح..,1
2,0,2,يوما واحدا»,يوما واحدا»,يوما واحدا»,1


In [5]:
COL_ORIGINAL = "L5"

# Drop rows with missing or empty original fragment
before = len(df_raw)
df_raw = df_raw.dropna(subset=[COL_ORIGINAL])
df_raw = df_raw[df_raw[COL_ORIGINAL].str.strip().str.len() > 0]
df_raw = df_raw.reset_index(drop=True)
print(f"Dropped {before - len(df_raw)} empty rows. Remaining: {len(df_raw)}")

Dropped 0 empty rows. Remaining: 20603


In [6]:
sentences = df_raw[COL_ORIGINAL]

In [7]:
def simplify_with_llm(sentence, target_level):
    level_instructions = {
        1: "بشكل بسيط جداً مناسب للأطفال، باستخدام كلمات قصيرة وشائعة فقط",
        2: "بشكل بسيط وواضح، مناسب للقراء المبتدئين",
        3: "بشكل مبسط قليلاً، مع الحفاظ على الأسلوب العربي الفصيح"
    }

    prompt = f"""أنت متخصص في تبسيط النصوص العربية. مهمتك تبسيط الجملة التالية {level_instructions[target_level]}.

قواعد مهمة:
- أعد كتابة الجملة فقط، بدون أي مقدمة أو شرح
- حافظ على المعنى الأصلي
- لا تضف معلومات جديدة
- اكتب جملة واحدة فقط

الجملة الأصلية: {sentence}

الجملة المبسّطة:"""

    response = requests.post(OLLAMA_URL, json={
        "model": "aya:8b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.3, "num_predict": 256}
    })
    return response.json()["response"].strip()

print("Ollama client ready.")

Ollama client ready.


In [8]:
def simplify_sentence_levels(sentence):
    results = {}
    for lvl in [1, 2, 3]:
        try:
            simplified = simplify_with_llm(sentence, lvl)
            results[lvl] = simplified
        except Exception as e:
            print(f"  Level {lvl} failed: {e}")
    return results if results else None

In [9]:
def is_valid(original, simplified):
    if not simplified or len(simplified.strip()) < 3:
        return False

    # Strip any residual prefixes
    prefixes_to_clean = ["الجملة المبسّطة:", "إليك", "أعد كتابة", "الترجمة:", "ببساطة:"]
    for p in prefixes_to_clean:
        simplified = simplified.replace(p, "").strip()

    if len(simplified) > len(original) * 2.5:
        return False

    if simplified.strip() == original.strip():
        return False

    return True

In [10]:
PART = "A"

all_rows = df_raw.to_dict("records")
print(f"Total SAMER fragments: {len(all_rows)}")

mid = len(all_rows) // 2
all_rows = all_rows[:mid] if PART == "A" else all_rows[mid:]
print(f"Part {PART}: {len(all_rows)} fragments")

# Resume from last checkpoint
checkpoint_file = f"/kaggle/input/datasets/naghamalshbrawi/samercheckpoint/samer_simplified_part{PART}_checkpoint_latest.csv"

if os.path.exists(checkpoint_file):
    simplified_data_grouped = pd.read_csv(checkpoint_file, dtype=str).to_dict("records")
    already_done = len(simplified_data_grouped)
    all_rows = all_rows[already_done:]
    print(f"Resuming from {already_done} fragments, {len(all_rows)} remaining")
else:
    simplified_data_grouped = []
    print("No checkpoint found, starting fresh")

SAVE_EVERY = 100

Total SAMER fragments: 20603
Part A: 10301 fragments
No checkpoint found, starting fresh


In [11]:
for count, row in enumerate(tqdm(all_rows)):
    sentence = row[COL_ORIGINAL]

    simplified_versions = simplify_sentence_levels(sentence)
    if not simplified_versions:
        continue

    # Wide row — original + all 3 levels + all metadata kept together
    entry = {"Original_Sentence": sentence}

    for lvl, simp in simplified_versions.items():
        clean = simp.strip()
        entry[f"Sentence_Level{lvl}"] = clean if is_valid(sentence, clean) else None

    # Carry over all SAMER metadata columns (book, para, split, existing levels, etc.)
    for col in row:
        if col != COL_ORIGINAL and col not in entry:
            entry[col] = row[col]

    entry["Source"] = "SAMER"

    simplified_data_grouped.append(entry)

    if (count + 1) % SAVE_EVERY == 0:
        pd.DataFrame(simplified_data_grouped).to_csv(
            f"samer_simplified_part{PART}_checkpoint_latest.csv", index=False
        )
        print(f"  Checkpoint saved: {len(simplified_data_grouped)} rows")

# Final save — catches the last <100 rows
pd.DataFrame(simplified_data_grouped).to_csv(
    f"samer_simplified_part{PART}_checkpoint_latest.csv", index=False
)
print(f"Done. Generated {len(simplified_data_grouped)} rows.")

  0%|          | 0/10301 [00:00<?, ?it/s]

  Checkpoint saved: 100 rows
  Checkpoint saved: 200 rows
  Checkpoint saved: 300 rows
  Checkpoint saved: 400 rows
  Checkpoint saved: 500 rows
  Checkpoint saved: 600 rows
  Checkpoint saved: 700 rows
  Checkpoint saved: 800 rows
  Checkpoint saved: 900 rows
  Checkpoint saved: 1000 rows
  Checkpoint saved: 1100 rows
  Checkpoint saved: 1200 rows
  Checkpoint saved: 1300 rows
  Checkpoint saved: 1400 rows
  Checkpoint saved: 1500 rows
  Checkpoint saved: 1600 rows
  Checkpoint saved: 1700 rows
  Checkpoint saved: 1800 rows
  Checkpoint saved: 1900 rows
  Checkpoint saved: 2000 rows
  Checkpoint saved: 2100 rows
  Checkpoint saved: 2200 rows
  Checkpoint saved: 2300 rows
  Checkpoint saved: 2400 rows
  Checkpoint saved: 2500 rows
  Checkpoint saved: 2600 rows
  Checkpoint saved: 2700 rows
  Checkpoint saved: 2800 rows
  Checkpoint saved: 2900 rows
  Checkpoint saved: 3000 rows
  Checkpoint saved: 3100 rows
  Checkpoint saved: 3200 rows
  Checkpoint saved: 3300 rows
  Checkpoint saved:

In [12]:
df = pd.DataFrame(simplified_data_grouped)
df.to_csv(f"samer_simplified_part{PART}_final.csv", index=False)
print(f"Saved {len(df)} rows to samer_simplified_part{PART}_final.csv")
print(df.head())

Saved 10301 rows to samer_simplified_partA_final.csv
                                   Original_Sentence  \
0                                       الفصل الثاني   
1                                       «وكان صباح..   
2                                        يوما واحدا»   
3  قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...   
4                     ينحدر على أحد جانبيه نهر جائش،   

                                     Sentence_Level1  \
0                                       الفصل التاني   
1                                      كان يوم جميل.   
2                                           يوم واحد   
3  إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم ق...   
4                    ينحدر نهرٌ على جانب هذا المكان.   

                                     Sentence_Level2  \
0                                               None   
1                                  كان يوماً جميلاً.   
2                                        يوما واحدا.   
3  قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قس

In [13]:
for entry in simplified_data_grouped[:5]:
    print("Original:  ", entry["Original_Sentence"])
    for lvl in [1, 2, 3]:
        label = {1: "Level 1 (simplest)", 2: "Level 2", 3: "Level 3"}[lvl]
        val = entry.get(f"Sentence_Level{lvl}")
        print(f"  {label}: {val if val else '[invalid]'}")
    print("-" * 60)

Original:   الفصل الثاني
  Level 1 (simplest): الفصل التاني
  Level 2: [invalid]
  Level 3: [invalid]
------------------------------------------------------------
Original:   «وكان صباح..
  Level 1 (simplest): كان يوم جميل.
  Level 2: كان يوماً جميلاً.
  Level 3: [invalid]
------------------------------------------------------------
Original:   يوما واحدا»
  Level 1 (simplest): يوم واحد
  Level 2: يوما واحدا.
  Level 3: يوماً واحداً.
------------------------------------------------------------
Original:   قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عميقة النوم إذا استثنينا حلما قصيرا ركب فيه جوادا بلا لجام جمح به في طريق وعر،
  Level 1 (simplest): إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم قصير ركب فيه جوادًا دون لجام وجد فيه نفسه في طريق وعر.
  Level 2: قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قسطاً وافراً من النوم. استثناء واحد فقط؛ حيث استيقظ على حلم قصير رأى فيه نفسه يركب جملاً بلا لجام في طريق وعر.
  Level 3: قضى إبراهيم، هذا هو اسمه، ليلة هادئة عميقة النوم، باستثناء حلم قصير 